# 02 Modelltraining

Ziel: Logistic Regression, Random Forest und HistGradientBoosting vergleichen. Bewertet wird mit Accuracy, Macro-F1 und Konfusionsmatrix.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
data_path_candidates = [
    Path("DryBeanDataset/Dry_Bean_Dataset.xlsx"),
    Path("../DryBeanDataset/Dry_Bean_Dataset.xlsx"),
]
data_path = next(path for path in data_path_candidates if path.exists())

df = pd.read_excel(data_path)
X_all = df.drop(columns="Class")
y = df["Class"]

selected_features = [
    "Area",
    "Perimeter",
    "AspectRation",
    "Compactness",
    "roundness",
    "ShapeFactor1",
    "ShapeFactor2",
    "ShapeFactor4",
]
X = X_all[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train.shape, X_test.shape

## Modelle

Skalierung wird fuer Logistic Regression verwendet. Baumverfahren benoetigen sie nicht zwingend.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    fitted_models[name] = model
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro"),
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]
y_pred = best_model.predict(X_test)

print(best_model_name)
print(classification_report(y_test, y_pred))

In [ ]:
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.title(f"Konfusionsmatrix: {best_model_name}")
plt.xlabel("Vorhersage")
plt.ylabel("Wahre Klasse")
plt.xticks(rotation=30)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()